# 🔐 pypdpg — getting started

**The data controller** holds personal data. **The data processor** has a
scoring model built on plain numpy. Today, running that model means handing
over plaintext. With `pypdpg`, the controller ships **ciphertext** instead —
and the processor's numpy code runs on it **unchanged**. (Controller and
processor in the GDPR / Thai PDPA sense — the exact relationship this tool
is built for.)

Encrypted in → encrypted out → only the controller can decrypt.

*Runs top-to-bottom on a fresh Colab runtime in under two minutes.*

In [ ]:
%%time
%pip install -q git+https://github.com/PDPG-lab/pypdpg

---
# 🏦 ACT 1 — The Data Controller: encrypt and ship

The controller holds 200 loan applications with five sensitive features.
Handing them to a processor in plaintext is exactly the kind of transfer
GDPR and PDPA make you justify. So it doesn't.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
N = 200

df = pd.DataFrame({
    "income":      rng.normal(58_000, 18_000, N).clip(18_000, 150_000),
    "debt":        rng.normal(22_000, 12_000, N).clip(0, 90_000),
    "age":         rng.uniform(21, 70, N),
    "tenure":      rng.uniform(0, 25, N),
    "utilization": rng.beta(2, 5, N),
})
df.head(3)

In [ ]:
import pypdpg as pdpg

ctx = pdpg.Context.create()          # CKKS: degree 16384, depth 4, ~128-bit security
ctx.save("controller.key")           # secret key — NEVER leaves the controller
ctx.save_public("processor.ctx")     # evaluation keys — safe to ship

pdpg.encrypt(df, ctx).save("data.enc")
print("shipped: data.enc + processor.ctx   (kept home: controller.key)")

In [ ]:
import os

plain_bytes = df.to_numpy().nbytes
enc_bytes = os.path.getsize("data.enc")
print(f"plaintext:     {plain_bytes / 1e3:8.1f} kB")
print(f"ciphertext:    {enc_bytes / 1e6:8.1f} MB   (~{enc_bytes / plain_bytes:.0f}x — the price of privacy today)")
print(f"processor.ctx: {os.path.getsize('processor.ctx') / 1e6:8.1f} MB   (one-time: the processor's evaluation keys)")

---
# 🏢 ACT 2 — The Data Processor: score blind

Everything below runs with **no secret key in the process**. Two files
arrived: `processor.ctx` and `data.enc`. Two lines of setup, then the
existing pipeline.

In [ ]:
import pypdpg as pdpg

pdpg.activate("processor.ctx")   # line 1: load the evaluation context
pdpg.install()                   # line 2: teach np.load about .enc files

### The processor's existing model code — untouched

In [ ]:
def score(X, w, b):
    """The processor's credit scorer. Written years ago. Knows nothing about encryption."""
    return X @ w + b

w = np.array([0.001, -0.002, 0.8, 3.0, -150.0])
b = 600.0

# sanity check on a plain sample applicant — ordinary floats in, ordinary floats out
sample = np.array([[60_000, 20_000, 35, 5.0, 0.3]])
print("plain sample score:", score(sample, w, b))

In [ ]:
X = np.load("data.enc")   # same call as always — named columns, encrypted values
X

In [ ]:
%%time
scores = score(X, w, b)   # the same function, now computing blind
scores

In [ ]:
# encrypted analytics keep their labels — and stay ciphertext
portfolio_means = X.mean()
portfolio_means.values.save("means.enc")
scores.save("result.enc")
portfolio_means

---
# 🏦 ACT 3 — The Data Controller: decrypt the verdict

`result.enc` comes home. Only `controller.key` can open it.

In [ ]:
pdpg.activate("controller.key")

result = pdpg.load("result.enc").decrypt()
true_scores = score(df.to_numpy(), w, b)    # plaintext ground truth, for honesty

print("first five scores:", np.round(result[:5], 2))
print(f"max abs error vs plaintext: {np.abs(result - true_scores).max():.2e}")
print("CKKS is approximate — off by ~1e-4 on ~600-point scores. Fine for scoring, and we say so.")
assert np.allclose(result, true_scores, atol=1e-2)

print()
print(pd.Series(pdpg.load("means.enc").decrypt(), index=df.columns).round(2))

---
# 🧨 And if the processor gets curious?

Every operation that would *reveal* something answers with a teaching error,
not a stack trace:

In [ ]:
pdpg.activate("processor.ctx")   # back in the processor's seat
X = np.load("data.enc")
try:
    X["income"] > 600
except pdpg.EncryptedOperationError as e:
    print(f"⛔ {e}")

The full tour of refusals — and the branchless patterns that compute the
same things *without* revealing them — lives in
[cookbook/03_branchless_logic](https://github.com/PDPG-lab/pypdpg/tree/main/demo/cookbook).

---
# What drops in, what needs a rewrite, what's coming

| | |
|---|---|
| ✅ **drop in now** | existing numpy/pandas code, unchanged: `+ - * /scalar` · `@` · `dot` · `sum` · `mean` · `square` · `**n` · sigmoid · named columns · fitted sklearn linear models · save/load · `np.load` |
| 🔁 **needs a rewrite** | data-dependent logic, written branchless (constant-time style), runs today: `if/else` → `gate*b + (1-gate)*c` · thresholds → `sigmoid` gates · filtering → full-shape masking |
| 🔜 **waiting on the engine** | exact comparisons, `max`/`sort` · ciphertext division · `exp`/`log`/`sqrt` · unlimited depth · encrypted@encrypted matmul · GPU. They'll drop in — your code won't change |

One thing fits no bucket, ever: *this* party **reading** the data. Not a
roadmap item — the security guarantee.

<sub>A [PDPG-lab](https://pdpglab.xyz) project. Current backend:
[TenSEAL](https://github.com/OpenMined/TenSEAL) (CKKS). More recipes in
[demo/cookbook](https://github.com/PDPG-lab/pypdpg/tree/main/demo/cookbook).</sub>